# Notebook that loads in .zarr data from ARCO ERA5 load-in
# and regrids it to cartesian coordinates


### Imports for regridding and data wrangling


In [1]:
import xesmf as xe   
import xarray as xr
import numpy as np
import os
from pathlib import Path


Load in zarr data

In [ ]:
ZARR_PATH = Path('../../Data/ERA5_FR_zarr/ERA5_FR.zarr/')

VARIABLES  = [
# 't2m', 
# 'd2m', 
'swvl1', 
'swvl2'
]

ds_raw = xr.merge(
    xr.open_zarr(ZARR_PATH / f'ERA5_FR_{var}_raw.zarr')
    for var in VARIABLES
)

/var/folders/p4/7cgg9ywn2hz45fzv3s976d_00000gn/T/ipykernel_2703/2027113673.py:10: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_raw = xr.merge(
/var/folders/p4/7cgg9ywn2hz45fzv3s976d_00000gn/T/ipykernel_2703/2027113673.py:10: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_raw = xr.merge(
/var/folders/p4/7cgg9ywn2hz45fzv3s976d_000

In [ ]:
ds_raw.time.dt.year.compute().values



In [ ]:
import xarray as xr
import scipy.spatial
import numpy as np

def mirror_point_at_360(ds):
    """
    Duplicate points at longitude=0 and place them at longitude=360.
    Prevents triangulation gaps at the wrap-around boundary.
    The longitude mask is computed eagerly since boolean Dask arrays
    cannot be used for indexing directly.
    """
    # Compute the boolean mask first before using it to index
    lon_mask = (ds.longitude == 0).compute()

    extra_point = (
        ds.where(lon_mask, drop=True)
          .assign_coords(longitude=lambda x: x.longitude + 360)
    )
    return xr.concat([ds, extra_point], dim='values')

def build_triangulation(x, y):
  grid = np.stack([x, y], axis=1)
  return scipy.spatial.Delaunay(grid)

def interpolate(data, tri, mesh):
  indices = tri.find_simplex(mesh)
  ndim = tri.transform.shape[-1]
  T_inv = tri.transform[indices, :ndim, :]
  r = tri.transform[indices, ndim, :]
  c = np.einsum('...ij,...j', T_inv, mesh - r)
  c = np.concatenate([c, 1 - c.sum(axis=-1, keepdims=True)], axis=-1)
  result = np.einsum('...i,...i', data[:, tri.simplices[indices]], c)
  return np.where(indices == -1, np.nan, result)

In [ ]:
# Mirror points at longitude=0 to prevent triangulation gaps at domain edge
ds_mirrored = mirror_point_at_360(ds_raw)

# Extract source grid coordinates as plain NumPy — triangulation needs these
src_lon = ds_mirrored.longitude.values
src_lat = ds_mirrored.latitude.values

print(f"Source points after mirroring: {len(src_lon)}")

In [ ]:
# This is the expensive step (~10-30 seconds) but only runs once
# The source grid geometry is identical for all variables and all timesteps
print("Building triangulation...")
tri = build_triangulation(src_lon, src_lat)
print("Done.")


In [ ]:
src_lon = ds_mirrored.longitude.compute().values
src_lat = ds_mirrored.latitude.compute().values

LAT_MIN, LAT_MAX = float(src_lat.min()), float(src_lat.max())
LON_MIN, LON_MAX = float(src_lon.min()), float(src_lon.max())

print(f"Latitude  bounds: {LAT_MIN} – {LAT_MAX}")
print(f"Longitude bounds: {LON_MIN} – {LON_MAX}")

lat_target = np.arange(LAT_MIN, LAT_MAX + 0.25, 0.25).round(2)
lon_target = np.arange(LON_MIN, LON_MAX + 0.25, 0.25).round(2)

print(f"\nTarget grid: {len(lat_target)} lats × {len(lon_target)} lons "
      f"= {len(lat_target)*len(lon_target)} points")

# Build (lon, lat) pairs for every point on the target grid
# interpolate() expects shape (n_target_points, 2)
lon_grid, lat_grid = np.meshgrid(lon_target, lat_target)
target_mesh = np.stack([lon_grid.ravel(), lat_grid.ravel()], axis=1)


In [ ]:
# ── Convert source coordinates to -180/180 before triangulation ───────────────

src_lon_raw = ds_mirrored.longitude.compute().values.copy()
src_lat     = ds_mirrored.latitude.compute().values

# Convert any longitudes > 180 to negative (0-360 → -180/180)
src_lon = np.where(src_lon_raw > 180, src_lon_raw - 360, src_lon_raw)

print(f"Longitude range after conversion: {src_lon.min():.2f} – {src_lon.max():.2f}")
print(f"Latitude range                  : {src_lat.min():.2f} – {src_lat.max():.2f}")

# ── Rebuild triangulation with corrected coordinates ──────────────────────────
print("Building triangulation...")
tri = build_triangulation(src_lon, src_lat)
print("Done.")

# ── Redefine target grid in -180/180 convention ───────────────────────────────
LAT_MIN = float(src_lat.min())
LAT_MAX = float(src_lat.max())
LON_MIN = float(src_lon.min())
LON_MAX = float(src_lon.max())

lat_target = np.arange(LAT_MIN, LAT_MAX + 0.25, 0.25).round(2)
lon_target = np.arange(LON_MIN, LON_MAX + 0.25, 0.25).round(2)

print(f"\nTarget grid: {len(lat_target)} lats × {len(lon_target)} lons")
print(f"Lat: {lat_target[0]} – {lat_target[-1]}")
print(f"Lon: {lon_target[0]} – {lon_target[-1]}")

# ── Rebuild target mesh ───────────────────────────────────────────────────────
lon_grid, lat_grid = np.meshgrid(lon_target, lat_target)
target_mesh = np.stack([lon_grid.ravel(), lat_grid.ravel()], axis=1)

# Always test on one slice before committing to a 44-year loop
test_slice = ds_mirrored['swvl1'].isel(time=0).values  # shape: (n_values,)

test_result = interpolate(
    test_slice[np.newaxis],   # needs shape (1, n_values)
    tri,
    target_mesh
).reshape(len(lat_target), len(lon_target))

# Quick visual check
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.pcolormesh(lon_target, lat_target, test_result, cmap='YlOrBr_r')
plt.colorbar(label='swvl1 (m³/m³)')
plt.title('Regrid test — single timestep')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.show()

print("Result shape:", test_result.shape)
print("NaN fraction:", np.isnan(test_result).mean().round(3))


In [ ]:
# ── Assign corrected -180/180 longitudes to ds_mirrored BEFORE the loop ──────
# src_lon was already converted to -180/180 when building the triangulation.
# ds_mirrored must carry the same convention so point ordering stays consistent.
ds_mirrored = ds_mirrored.assign_coords(
    longitude=('values', src_lon)   # src_lon already converted to -180/180
)

# Sanity check
print(f"ds_mirrored longitude range: {float(ds_mirrored.longitude.min()):.2f} – "
                                    f"{float(ds_mirrored.longitude.max()):.2f}")

In [ ]:
years_available = np.unique(ds_mirrored.time.dt.year.compute().values)
year_start = int(years_available.min())
year_end   = int(years_available.max())

OUT_PATH = Path('../../Data/ERA5_FRBENLDE_zarr/ERA5_FRBENLDE_regridded.zarr/')

for var in VARIABLES:
    out_zarr = OUT_PATH / f'ERA5_FRBENLDE_{var}.zarr'

    if (out_zarr / '.zmetadata').exists():
        print(f"{var}: already exists, skipping.")
        continue

    print(f"\nRegridding {var}...")
    da_var = ds_mirrored[var]

    all_times = da_var.time.compute().values
    all_years = all_times.astype('datetime64[Y]').astype(int) + 1970

    yearly_results = []

    for year in range(year_start, year_end + 1):
        year_mask = all_years == year
        if not year_mask.any():
            print(f"  {year} (no data, skipping)", end=' ', flush=True)
            continue

        print(f"  {year}", end=' ', flush=True)

        da_year = da_var.isel(time=year_mask).compute()
        n_time  = da_year.sizes['time']

        regridded = np.full(
            (n_time, len(lat_target), len(lon_target)),
            np.nan, dtype=np.float32
        )

        for t in range(n_time):
            # .values extracts the raw data array — ordering matches src_lon/src_lat
            # used to build the triangulation, so interpolation is spatially correct
            flat = da_year.isel(time=t).values
            regridded[t] = interpolate(
                flat[np.newaxis], tri, target_mesh
            ).reshape(len(lat_target), len(lon_target))

        yearly_results.append(xr.DataArray(
            regridded,
            dims=['valid_time', 'latitude', 'longitude'],
            coords={
                'valid_time': da_year.time.values,
                'latitude'  : lat_target,
                'longitude' : lon_target,   # -180/180 convention throughout
            },
            attrs=da_year.attrs
        ))

    print(f"\n  Saving {var}...")
    (
        xr.concat(yearly_results, dim='valid_time')
          .to_dataset(name=var)
          .chunk({'valid_time': 365, 'latitude': -1, 'longitude': -1})
          .to_zarr(out_zarr, mode='w')
    )
    print(f"  Done → {out_zarr}")

print("\nAll variables regridded.")

In [ ]:
ds_mirrored.swvl1.isel(time=0).plot()